In [81]:
import os
import mlflow
import mlflow.pytorch
import json

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from mlflow.tracking import MlflowClient

In [82]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
path = f"{PROJECT_ROOT}/mlruns"

os.makedirs(os.path.join(path, ".trash"), exist_ok=True)

mlflow.set_tracking_uri(f"file://{path}")
mlflow.set_experiment("spaceship_titanic")

<Experiment: artifact_location=('file:///Users/admin/DataSince/Kaggle competitions/Spaceship '
 'Titanic/mlruns/689233284165520319'), creation_time=1767104868196, experiment_id='689233284165520319', last_update_time=1767104868196, lifecycle_stage='active', name='spaceship_titanic', tags={'mlflow.experimentKind': 'custom_model_development'}>

# Blending without proba_oof

In [83]:
run_name = "blending_pytorch_add_hp_catboost_fix_optuna_function_without_proba_oof"

with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id
client = MlflowClient() 
print(run_id)

3603ae2277614701a9202be15f944d31


In [84]:
preds_list = []
auc_list = []
tresh_list = []

In [85]:
# Предикт 1
old_run_name_1 = "pytorch_cv5_optuna_52features_add_hp"

path = "../predictions/proba_submission_16.csv"
df = pd.read_csv(path)
pred = df["Transported"]
preds_list.append(pred)
print(pred.shape)
# для взвешивания испфользую cv_auc_mean
auc = 0.88
test_acc = 0.79845
auc_list.append(auc)
best_thr_1 = 0.543
tresh_list.append(best_thr_1)

(4277,)


In [86]:
client.log_param(run_id, "src_run_1_name", old_run_name_1)
client.log_param(run_id, "src_run_1_pred_file", path)
client.log_param(run_id, "src_run_1_auc", auc)
client.log_param(run_id, "src_run_1_test_acc", test_acc)
client.log_param(run_id, "src_run_1_best_thr", best_thr_1);    

In [87]:
# Предикт 2
old_run_name_2 = "catboost_optuna_cv5_fix_optuna_function"
path = "../predictions/proba_submission_14.csv"
df = pd.read_csv(path)
pred = df["Transported"]
preds_list.append(pred)
print(pred.shape)
auc = 0.911
test_acc = 0.80547
auc_list.append(auc)
best_thr_2 = 0.53
tresh_list.append(best_thr_2)

(4277,)


In [88]:
client.log_param(run_id, "src_run_2_name", old_run_name_2)
client.log_param(run_id, "src_run_2_pred_file", path)
client.log_param(run_id, "src_run_2_auc", auc)
client.log_param(run_id, "src_run_2_test_acc", test_acc)
client.log_param(run_id, "src_run_2_best_thr", best_thr_2 );

In [89]:
preds_list

[0       0.264472
 1       0.005603
 2       0.996295
 3       0.949340
 4       0.617080
           ...   
 4272    0.749816
 4273    0.577932
 4274    0.935589
 4275    0.767608
 4276    0.908853
 Name: Transported, Length: 4277, dtype: float64,
 0       0.607576
 1       0.016698
 2       0.996151
 3       0.987510
 4       0.603344
           ...   
 4272    0.673721
 4273    0.501611
 4274    0.964906
 4275    0.782044
 4276    0.677261
 Name: Transported, Length: 4277, dtype: float64]

In [90]:
tresh_list

[0.543, 0.53]

In [91]:
auc_arr = np.array(auc_list, dtype="float32")
weights = auc_arr / auc_arr.sum()
print("weights", weights)
preds_arr = np.stack(preds_list, axis=0)  # (n_folds, n_test)

# взвешенное среднее по фолдам
test_proba_weighted = np.average(
    preds_arr,
    axis=0,
    weights=weights
)
test_proba_weighted

weights [0.4913456 0.5086544]


array([0.43899331, 0.01124645, 0.99622193, ..., 0.95050154, 0.77495075,
       0.79105271])

In [92]:
thr_arr = np.array(tresh_list, dtype="float32")

# взвешенный порог
thr_weighted = np.average(
    thr_arr,
    axis=0,
    weights=weights
)
client.log_param(run_id, "thr_weighted", thr_weighted)
print("tresh_weighted", thr_weighted)

# Так как у нас нет proba_oof(все предикты на валидации) и я не уверен в звешенном пороге,
# то проверю все три порога
test_pred_bool_1 = (test_proba_weighted >= best_thr_1).astype(bool)
test_pred_bool_2 = (test_proba_weighted >= best_thr_2).astype(bool)
test_pred_bool_weighted = (test_proba_weighted >= thr_weighted).astype(bool)
test_pred_bool_1

tresh_weighted 0.5363875


array([False, False,  True, ...,  True,  True,  True])

In [93]:
df_test = pd.read_csv('../data/processed/test_3')
df_test.shape

(4277, 54)

In [94]:
# логируем вероятности в MLflow и сохраняем на диск
proba = df_test[["PassengerId"]].copy() 
proba["Transported"] = test_proba_weighted
proba_path = "../predictions/proba_submission_18.csv"
proba.to_csv(proba_path, index=False)
client.log_artifact(run_id, proba_path, artifact_path="submissions")
proba.shape

(4277, 2)

In [95]:
# логируем мтки классов в MLflow и сохраняем на диск   test_pred_bool_1
submision = df_test[["PassengerId"]].copy() 
submision["Transported"] = test_pred_bool_1
sub_path = "../predictions/submission_18_1.csv"
submision.to_csv(sub_path, index=False)
client.log_artifact(run_id, sub_path, artifact_path="submissions")

submision.shape

(4277, 2)

In [96]:
# логируем мтки классов в MLflow и сохраняем на диск  test_pred_bool_2
submision = df_test[["PassengerId"]].copy() 
submision["Transported"] = test_pred_bool_2
sub_path = "../predictions/submission_18_2.csv"
submision.to_csv(sub_path, index=False)
client.log_artifact(run_id, sub_path, artifact_path="submissions")

submision.shape

(4277, 2)

In [97]:
# логируем мтки классов в MLflow и сохраняем на диск  test_pred_bool_weighted
submision = df_test[["PassengerId"]].copy() 
submision["Transported"] = test_pred_bool_weighted
sub_path = "../predictions/submission_18_weighted.csv"
submision.to_csv(sub_path, index=False)
client.log_artifact(run_id, sub_path, artifact_path="submissions")

submision.shape

(4277, 2)

In [98]:
# Из трёх метрик на тесте я выбиру лучшую, полученную взвешенным порогом
test_acc = 0.80593 # скопировал из Kaggle

client.log_metric(run_id, "test_acc", float(test_acc))